# 06 — Final linear-Q controlled optimization and stop-line decision

This notebook records the final bounded optimization attempt. The original agent is preserved. Six enhanced agents start from the same 61,973-update checkpoint and add eight explicit Task-4 interaction features.

## Design

New features: safe crate bomb, safe opponent bomb, narrow bomb escape, danger × low freedom, safe opponent progress, adjacent enemy, nearby enemy, and late-hunt progress. The factorial sweep is `alpha ∈ {0.003, 0.001}` × `n_step ∈ {3,5,8}`. Every candidate trained for 200 rounds against three rule agents. Validation used the same 50-round unseen seed 54001 with zero exploration.

Promotion thresholds were fixed before evaluation: mean score ≥3.7, suicides ≤0.25/round, survival ≥55%, with no Task-1/2 regression.

## 50-round validation results

| alpha | n-step | score | win | suicides | survival | result |
|---:|---:|---:|---:|---:|---:|---|
| .003 | 3 | 1.86 | .18 | .44 | .54 | fail |
| .003 | 5 | 1.38 | .08 | .26 | .66 | fail |
| .003 | 8 | **3.32** | **.30** | **.22** | **.64** | fail: score |
| .001 | 3 | 2.80 | .32 | .22 | .72 | fail: score |
| .001 | 5 | 2.74 | .24 | .38 | .38 | fail |
| .001 | 8 | 1.34 | .08 | .68 | .18 | fail |

No candidate passed all gates. Alpha=.003/n-step=8 was frozen as the closest candidate and advanced to the required regression and locked tests.

In [ ]:
from pathlib import Path
from evaluation.summarize_stats import summarize_match
for name in ['linear_q_a003_n3','linear_q_a003_n5','linear_q_a003_n8',
             'linear_q_a001_n3','linear_q_a001_n5','linear_q_a001_n8']:
    path = Path(f'results/{name}_validation50.json')
    print(next(row for row in summarize_match([path]) if row['agent'] == name))

## Regression and paired 100-round locked test

The selected checkpoint retained Task 1 (50.0 coins, 100% survival) and improved Task 2 to 5.95 score and 75.5 crates/round with zero suicide.

| locked seed 55001 | score | win | suicides | survival | invalid | crates |
|---|---:|---:|---:|---:|---:|---:|
| original linear-Q | **3.02** | .26 | .43 | .34 | .68 | 18.83 |
| enhanced frozen | 2.98 | **.30** | **.29** | **.62** | **.12** | **28.35** |

The enhanced model substantially improves safety, survival, legality, crates, rank, and score margin, but its mean score remains flat and fails both the 3.7 score gate and the 0.25 suicide gate. The predeclared stop line is therefore triggered: further linear-Q tuning is not justified as the main competition path. Keep `frozen_linear_q_final` as a stronger safety-oriented interpretable baseline and prioritize nonlinear architectures for Task 4.